# Healthcare Data Understanding, Cleaning & Exploratory Analysis

This notebook is written for beginners.

We will:
- understand the healthcare dataset
- clean missing medical codes
- convert the date columns
- calculate hospital stay in days
- categorize admissions as Emergency, Elective, or Urgent
- summarize billing amounts and hospital stays
- look at patient demographics for each medical condition


## Step 1: Load the libraries

`pandas` helps us work with tables and data.

`matplotlib` helps us create simple charts.

Before running the notebook, upload `healthcare_raw.csv` to Google Colab.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Step 2: Load the dataset

In [ ]:
df = pd.read_csv("healthcare_raw.csv")

df.head()

## Step 3: Understand the dataset

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns)

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

We can see that some values in `Medical_Code` are missing.

We will replace those missing values with `Unknown`.

In [ ]:
df["Medical_Code"] = df["Medical_Code"].fillna("Unknown")

print(df["Medical_Code"].isnull().sum())

## Step 4: Standardize the date columns

In [ ]:
df["Admission_Date"] = pd.to_datetime(
    df["Admission_Date"],
    dayfirst=True,
    errors="coerce"
)

df["Discharge_Date"] = pd.to_datetime(
    df["Discharge_Date"],
    dayfirst=True,
    errors="coerce"
)

df[["Admission_Date", "Discharge_Date"]].head()

In [ ]:
print(df[["Admission_Date", "Discharge_Date"]].dtypes)

Now the dates are stored as date values instead of normal text.

This lets us calculate the number of days a patient stayed in the hospital.

In [ ]:
df["Stay_Days"] = (
    df["Discharge_Date"] - df["Admission_Date"]
).dt.days

df[["Admission_Date", "Discharge_Date", "Stay_Days"]].head()

In [ ]:
print("Minimum stay:", df["Stay_Days"].min(), "days")
print("Maximum stay:", df["Stay_Days"].max(), "days")

## Step 5: Categorize admissions by urgency

In [ ]:
print(df["Admission_Type"].value_counts())

The dataset uses `Routine` for normal admissions.

For this task, we will treat `Routine` as `Elective`.

In [ ]:
df["Admission_Type"] = (
    df["Admission_Type"]
    .astype(str)
    .str.strip()
    .replace({"Routine": "Elective"})
)

df["Urgency"] = df["Admission_Type"]

df["Urgency"].value_counts()

In [ ]:
print("Urgency categories:")
print(df["Urgency"].unique())

## Step 6: Summary statistics for billing amounts

In [ ]:
print("Average billing amount:", df["Billing_Amount"].mean())
print("Median billing amount:", df["Billing_Amount"].median())
print("Minimum billing amount:", df["Billing_Amount"].min())
print("Maximum billing amount:", df["Billing_Amount"].max())

In [ ]:
df["Billing_Amount"].describe()

## Step 7: Summary statistics for hospital stays

In [ ]:
print("Average hospital stay:", df["Stay_Days"].mean(), "days")
print("Median hospital stay:", df["Stay_Days"].median(), "days")
print("Minimum hospital stay:", df["Stay_Days"].min(), "days")
print("Maximum hospital stay:", df["Stay_Days"].max(), "days")

In [ ]:
df["Stay_Days"].describe()

## Step 8: Segment patient demographics by medical condition

In [ ]:
condition_summary = df.groupby("Medical_Condition").agg(
    Patient_Count=("Patient_ID", "count"),
    Average_Age=("Age", "mean")
).sort_values("Patient_Count", ascending=False)

condition_summary

### Gender distribution for each medical condition

In [ ]:
gender_by_condition = pd.crosstab(
    df["Medical_Condition"],
    df["Gender"]
)

gender_by_condition

### Average billing and hospital stay by medical condition

In [ ]:
condition_costs = df.groupby("Medical_Condition").agg(
    Average_Billing=("Billing_Amount", "mean"),
    Average_Stay_Days=("Stay_Days", "mean")
).sort_values("Average_Billing", ascending=False)

condition_costs

## Step 9: Simple charts

In [ ]:
urgency_counts = df["Urgency"].value_counts()

urgency_counts.plot(kind="bar")
plt.title("Number of Admissions by Urgency")
plt.xlabel("Urgency")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.show()

In [ ]:
condition_counts = df["Medical_Condition"].value_counts().sort_values()

condition_counts.plot(kind="barh", figsize=(8, 8))
plt.title("Patients by Medical Condition")
plt.xlabel("Number of Patients")
plt.ylabel("Medical Condition")
plt.show()

In [ ]:
df["Billing_Amount"].plot(kind="hist", bins=10)
plt.title("Distribution of Billing Amounts")
plt.xlabel("Billing Amount")
plt.ylabel("Number of Patients")
plt.show()

## Step 10: Final healthcare summary

In [ ]:
average_bill = df["Billing_Amount"].mean()
average_stay = df["Stay_Days"].mean()
most_common_condition = df["Medical_Condition"].value_counts().idxmax()
most_common_urgency = df["Urgency"].value_counts().idxmax()

print("Healthcare Data Summary")
print("-----------------------")
print("Total patients:", len(df))
print("Average billing amount:", round(average_bill, 2))
print("Average hospital stay:", round(average_stay, 2), "days")
print("Most common medical condition:", most_common_condition)
print("Most common admission category:", most_common_urgency)

print("\nThe dataset is now cleaned and ready for further analysis.")

## Conclusion

We cleaned the missing medical codes, converted the date columns into proper date format, and calculated hospital stay in days.

We also changed `Routine` admissions to `Elective` so the admission categories match the task.

Finally, we calculated basic statistics for billing and hospital stays and compared patient demographics across medical conditions.